# שלב 08 — Node2Vec: Graph Learning ו-Node Embeddings

**Node2Vec** לומד ייצוג וקטורי (embedding) לכל תחנה על‑ידי 'הליכות אקראיות' על הגרף. תחנות בעלות תפקיד מבני דומה מקבלות וקטורים דומים. עם ה‑embeddings נבדוק:

1. האם ניתן לנבא תחנה קריטית מהווקטור שלה (סיווג).
2. האם המבנה הנלמד משמר הפרדה גיאוגרפית (t-SNE / PCA).

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas numpy networkx matplotlib seaborn python-bidi node2vec scikit-learn

In [ ]:
from pathlib import Path
import pickle, json
import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
BASE = ROOT / "public_transport_network_notebooks"
GRAPH_DIR = BASE / "outputs" / "02_graph_construction"
METRICS_CSV = BASE / "outputs" / "04_centrality_analysis" / "stop_metrics.csv"
AP_CSV = BASE / "outputs" / "03_network_descriptive_analysis" / "articulation_points.csv"
COMM_CSV = BASE / "outputs" / "07_community_detection" / "community_assignments.csv"
OUT_DIR = BASE / "outputs" / "08_graph_learning_node_embeddings"
FIG_DIR = BASE / "figures" / "08_graph_learning_node_embeddings"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIM, WALK_LENGTH, NUM_WALKS = 64, 20, 5
print("OUT_DIR:", OUT_DIR)

In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from bidi.algorithm import get_display


def _fix(t):
    """מסדר טקסט עברי לתצוגה נכונה (bidi). אנגלית ומספרים נשארים כמו שהם."""
    if isinstance(t, str) and any(0x590 <= ord(c) <= 0x5FF for c in t):
        return get_display(t)
    return t


import matplotlib.text as _mt
if not getattr(_mt.Text, "_bidi", False):
    _orig = _mt.Text.set_text
    def _set(self, s):
        if isinstance(s, str) and getattr(self, "_disp", None) == s:
            return _orig(self, s)
        f = _fix(s)
        if isinstance(f, str):
            self._disp = f
        return _orig(self, f)
    _mt.Text.set_text = _set
    _mt.Text._bidi = True

sns.set_theme(style="whitegrid", font_scale=1.1)
matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("עברית בגרפים מופעלת")

## אימון Node2Vec

מאמנים על הרכיב הקשור הגדול (לחיסכון בזמן), 64 ממדים, seed=42. **התא עשוי לקחת מספר דקות.**

In [ ]:
with open(GRAPH_DIR / "graph_undirected.pkl", "rb") as f:
    G = pickle.load(f)
metrics = pd.read_csv(METRICS_CSV, encoding="utf-8-sig")
metrics["betweenness"] = pd.to_numeric(metrics["betweenness"], errors="coerce").fillna(0)
ap_df = pd.read_csv(AP_CSV, encoding="utf-8-sig")
comm_df = pd.read_csv(COMM_CSV, encoding="utf-8-sig") if COMM_CSV.exists() else pd.DataFrame()

Gc = G.subgraph(max(nx.connected_components(G), key=len)).copy()
print(f"Node2Vec על הרכיב הגדול: {Gc.number_of_nodes():,} צמתים")
n2v = Node2Vec(Gc, dimensions=EMBEDDING_DIM, walk_length=WALK_LENGTH, num_walks=NUM_WALKS,
               p=1, q=1, workers=1, seed=42, quiet=True)
model = n2v.fit(window=5, min_count=1, batch_words=4, epochs=3)
embeddings = {node: model.wv[str(node)] for node in Gc.nodes()}

emb_df = pd.DataFrame([{"stop_id": k, **{f"e{i}": v for i, v in enumerate(vec)}} for k, vec in embeddings.items()])
emb_df.to_csv(OUT_DIR / "embeddings_df.csv", index=False, encoding="utf-8-sig")
print(f"Embeddings: {len(embeddings)} צמתים x {EMBEDDING_DIM} ממדים")

## סיווג תחנות קריטיות

מאמנים מסווגים (Logistic Regression, Random Forest) לנבא אם תחנה קריטית מתוך ה‑embedding בלבד. מודדים ב‑F1 כי הקריטיות נדירה (~2.5%). תוצאה נמוכה כאן היא ממצא מעניין: קריטיות מבנית קשה לנבא מהליכות אקראיות מקומיות.

In [ ]:
ap_set = set(ap_df["stop_id"].astype(str))
btw_p90 = metrics["betweenness"].quantile(0.90)
metrics_idx = metrics.set_index("stop_id")

ids = list(embeddings.keys())
X, y = [], []
for n in ids:
    X.append(embeddings[n])
    is_ap = str(n) in ap_set
    is_btw = metrics_idx.loc[n, "betweenness"] >= btw_p90 if n in metrics_idx.index else False
    y.append(1 if (is_ap or is_btw) else 0)
X, y = np.array(X), np.array(y)
print(f"קריטיות: {sum(y)} / {len(y)} ({sum(y)/len(y)*100:.1f}%)")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr); X_te = scaler.transform(X_te)

results = {}
for name, clf in [("LogisticRegression", LogisticRegression(class_weight="balanced", max_iter=500)),
                  ("RandomForest", RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42))]:
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    f1 = f1_score(y_te, y_pred, average="binary")
    results[name] = {"f1": round(f1, 4), "confusion": confusion_matrix(y_te, y_pred).tolist()}
    print(f"{name}: F1={f1:.4f}")
    print(classification_report(y_te, y_pred, target_names=["רגילה", "קריטית"]))

with open(OUT_DIR / "classification_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
critical_labels = dict(zip(ids, y.tolist()))

## גרפים: PCA וקונפוזיון

הטמעות אחרי PCA, צבועות לפי קריטיות, ומטריצות בלבול לכל מסווג.

In [ ]:
matrix = np.array([embeddings[n] for n in ids])
pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(matrix)
from matplotlib.patches import Patch
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(emb_2d[:, 0], emb_2d[:, 1], s=4, alpha=0.4,
           c=["#94a3b8" if critical_labels[n] == 0 else "#dc2626" for n in ids])
ax.legend(handles=[Patch(color="#94a3b8", label="רגילה"), Patch(color="#dc2626", label="קריטית")])
ax.set_title(f"PCA של Node Embeddings (שונות: {sum(pca.explained_variance_ratio_)*100:.1f}%)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
plt.tight_layout()
plt.savefig(FIG_DIR / "pca_embeddings.png", dpi=150)
plt.show()

for name, res in results.items():
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(np.array(res["confusion"]), annot=True, fmt="d", cmap="Blues",
                xticklabels=["רגילה", "קריטית"], yticklabels=["רגילה", "קריטית"], ax=ax)
    ax.set_xlabel("ניבוי"); ax.set_ylabel("אמת")
    ax.set_title(f"Confusion Matrix — {name} (F1={res['f1']:.3f})")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"confusion_matrix_{name.lower()[:2]}.png", dpi=150)
    plt.show()